# 🎨 Embeddings & Semantic Similarity Visualization

**Estimated time: 12-15 minutes**

## What are Embeddings?

Embeddings are numerical representations of text (words, sentences, or documents) in a high-dimensional vector space. They capture **semantic meaning** - words or sentences with similar meanings are positioned close to each other in this space.

Think of it like a map where:
- Each word/sentence is a point
- Similar concepts cluster together
- Distance measures similarity

### Why are they useful?

- 🔍 **Semantic Search**: Find similar documents based on meaning, not just keywords
- 📊 **Clustering**: Group similar items together
- 🤖 **Transfer Learning**: Use pre-trained models for downstream tasks
- 🎯 **Recommendation Systems**: Find similar products, articles, or content

### How do we measure similarity?

**Cosine Similarity** measures the angle between two vectors:
- Score of 1.0 = identical meaning
- Score of 0.0 = unrelated
- Score of -1.0 = opposite (rare in practice)

Let's dive in! 🚀

## 📦 Setup & Installation

In [1]:
# Import libraries
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


## 🤖 Load Embedding Model

We'll use **all-MiniLM-L6-v2**, a compact but powerful model that:
- Generates 384-dimensional embeddings
- Is fast and efficient
- Works great for semantic similarity tasks

In [91]:
# Load the model (this may take a minute on first run)
model = SentenceTransformer('all-MiniLM-L6-v2')

print(f"✅ Model loaded!");
print(f"   Embedding dimension: {model.get_sentence_embedding_dimension()}");

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded!
   Embedding dimension: 384


## 🔢 Example 1: Word Embeddings & Similarity

Let's start simple by comparing individual words.

In [3]:
# Define some words
words = ['king', 'queen', 'man', 'woman', 'prince', 'princess', 
         'dog', 'cat', 'puppy', 'kitten',
         'car', 'vehicle', 'bicycle', 'motorcycle']

# Generate embeddings
word_embeddings = model.encode(words)

print(f"Generated {len(word_embeddings)} embeddings")
print(f"Each embedding has {len(word_embeddings[0])} dimensions")
print(f"\nExample - 'king' embedding (first 10 values):")
print(word_embeddings[0][:10])

Generated 14 embeddings
Each embedding has 384 dimensions

Example - 'king' embedding (first 10 values):
[-0.05959932  0.05051232 -0.06951012  0.07968019 -0.04674765  0.00098894
  0.07904322 -0.01273938  0.05839579 -0.03140246]


We can compare the "similarity" of words, according to our embedding mechanism, by computing their cosine similarity

Exercise: Compare pairs of words using their embeddings.

Remember Conine similarity of 1.0 = most similar
cosine similarity of 0.0 is least similar

In [20]:
# hint:
dict_embeddings = {word: embedding for word, embedding in zip(words, word_embeddings)}
dict_embeddings["king"]
word_a = dict_embeddings["king"].reshape(1, -1)
word_b = dict_embeddings["queen"].reshape(1, -1)

# similarity
float(cosine_similarity(word_a, word_b)[0, 0])


0.6807125806808472

In [56]:
# Calculate similarity between all words
similarity_matrix = cosine_similarity(word_embeddings)

# Create an interactive heatmap
fig = go.Figure(data=go.Heatmap(
    z=similarity_matrix,
    x=words,
    y=words,
    colorscale='RdYlGn',
    texttemplate='%{z:.2f}',
    textfont={"size": 10},
    hovertemplate='Similarity: %{z:.2f}<extra></extra>',
    hoverongaps=False,
    colorbar=dict(title="Similarity")
))

fig.update_layout(
    title='Word Similarity Heatmap (Cosine Similarity)',
    xaxis_title='Words',
    yaxis_title='Words',
    width=800,
    height=700
)

fig.show()

print("\n💡 Observations:")
print("   - Diagonal is 1.0 (words are identical to themselves)")
print("   - 'king' and 'queen' are very similar")
print("   - 'dog' and 'cat' are similar (both pets)")
print("   - 'king' and 'dog' are less similar (different concepts)")


💡 Observations:
   - Diagonal is 1.0 (words are identical to themselves)
   - 'king' and 'queen' are very similar
   - 'dog' and 'cat' are similar (both pets)
   - 'king' and 'dog' are less similar (different concepts)


## 🧮 Example 2: Semantic Arithmetic (Vector Math)

One of the coolest properties of embeddings: we can do **math** with meaning!

Famous example: `king - man + woman ≈ queen`

Let's test this!

In [89]:
def find_most_similar(target_embedding, word_embeddings, words, exclude_words=None, top_k=5):
    """Find the most similar words to a target embedding"""
    similarities = cosine_similarity([target_embedding], word_embeddings)[0]
    
    # Create list of (word, similarity) pairs
    word_similarities = list(zip(words, similarities))
    
    # Filter out excluded words
    if exclude_words:
        word_similarities = [(w, s) for w, s in word_similarities if w not in exclude_words]
    
    # Sort by similarity (descending)
    word_similarities.sort(key=lambda x: x[1], reverse=True)
    
    return word_similarities[:top_k]

# Get individual embeddings
king_emb = model.encode('king')
male_emb = model.encode('male')
female_emb = model.encode('female')
royal_emb = model.encode('royal')


# Perform vector arithmetic
result_emb = king_emb - male_emb + female_emb 

# Find what this new vector is closest to
candidates = ['queen', 'princess', 'man', 'prince', 'lady', 'monarch', 'duchess', 'empress']
candidate_embeddings = model.encode(candidates)

similar_words = find_most_similar(result_emb, candidate_embeddings, candidates, top_k=5)

print("🎯 Vector Arithmetic: king - man + woman = ?\n")
print("Most similar words:")
for i, (word, score) in enumerate(similar_words, 1):
    bar = '█' * int(score * 50)
    print(f"{i}. {word:15s} {bar} {score:.4f}")

print("\n✅ It works! The model understands the gender relationship!")

🎯 Vector Arithmetic: king - man + woman = ?

Most similar words:
1. queen           ███████████████████████████████████ 0.7048
2. monarch         █████████████████████████████████ 0.6639
3. lady            █████████████████████████ 0.5015
4. princess        █████████████████████████ 0.5008
5. empress         ██████████████████████ 0.4518

✅ It works! The model understands the gender relationship!


In [90]:
# Let's try more examples!
def semantic_analogy(word1, word2, word3, candidates):
    """Solve: word2 is to word1 as word3 is to ?"""
    emb1 = model.encode(word1)
    emb2 = model.encode(word2)
    emb3 = model.encode(word3)
    
    # word1 - word2 + word3
    result = emb1 - emb2 + emb3
    
    candidate_embeddings = model.encode(candidates)
    similar = find_most_similar(result, candidate_embeddings, candidates, 
                                exclude_words=[word1, word2, word3], top_k=3)
    
    print(f"\n{word1} - {word2} + {word3} = ?")
    print(f"Top predictions:")
    for word, score in similar:
        print(f"  → {word} ({score:.3f})")

# Test various analogies
print("🧪 Semantic Analogies Test:\n" + "="*50)

semantic_analogy('France', 'Paris', 'London', 
                ['England', 'Britain', 'UK', 'Germany', 'Italy', 'Spain', "France", "Wales"])

semantic_analogy('doctor', 'hospital', 'school', 
                ['caretaker', 'nurse', 'teacher', 'office', 'scientist', 'laboratory'])

semantic_analogy('puppy', 'dog', 'cat', 
                ['cat', 'feline', 'tiger', 'lion', 'pet', 'animal', 'kitten'])

🧪 Semantic Analogies Test:

France - Paris + London = ?
Top predictions:
  → Britain (0.782)
  → England (0.769)
  → UK (0.685)

doctor - hospital + school = ?
Top predictions:
  → teacher (0.613)
  → scientist (0.550)
  → office (0.338)

puppy - dog + cat = ?
Top predictions:
  → kitten (0.828)
  → feline (0.613)
  → pet (0.590)


## 🔍 Exercise: Semantic Search

Now let's do something practical: **search research papers by semantic meaning**!

We'll create a mini research paper database and find similar papers based on their titles.

In [64]:
# Sample research paper titles (diverse topics)
papers = [
    "Deep Learning for Image Classification Using Convolutional Neural Networks",
    "Neural Networks for Computer Vision: A Comprehensive Review",
    "Attention Mechanisms in Natural Language Processing",
    "BERT: Pre-training of Deep Bidirectional Transformers",
    "Quantum Computing: Algorithms and Applications",
    "Quantum Error Correction and Fault-Tolerant Computing",
    "Climate Change Impact on Marine Ecosystems",
    "Ocean Acidification and Coral Reef Degradation",
    "Machine Learning in Healthcare: Predictive Analytics",
    "AI-Based Disease Diagnosis Using Medical Imaging",
    "Blockchain Technology for Secure Financial Transactions",
    "Cryptocurrency Mining and Environmental Sustainability",
    "Gene Editing with CRISPR-Cas9: Ethical Implications",
    "Synthetic Biology and Genetic Engineering Advances",
    "Renewable Energy Storage Solutions for Smart Grids",
    "Solar Panel Efficiency and Energy Conversion Optimization"
]

# Generate embeddings for all papers
paper_embeddings = model.encode(papers)

print(f"✅ Indexed {len(papers)} research papers\n")

✅ Indexed 16 research papers



In [65]:
def semantic_search(query, papers, paper_embeddings, top_k=5):
    """Search for papers similar to the query
    Hint: use the cosine similarity
    
    """
    # Encode the query
    query_embedding = model.encode(query)
    
    # Calculate similarities
    similarities = cosine_similarity([query_embedding], paper_embeddings)[0]
    
    # Get top results
    top_indices = np.argsort(similarities)[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        results.append({
            'paper': papers[idx],
            'similarity': similarities[idx]
        })
    return results
    

# Test semantic search
queries = [
    "transformers for text understanding",
    "environmental impact of oceans",
    "medical AI applications"
]

for query in queries:
    print(f"\n🔍 Query: '{query}'")
    print("="*80)
    
    results = semantic_search(query, papers, paper_embeddings, top_k=3)
    
    for i, result in enumerate(results, 1):
        score_bar = '█' * int(result['similarity'] * 40)
        print(f"\n{i}. [Score: {result['similarity']:.3f}] {score_bar}")
        print(f"   {result['paper']}")


🔍 Query: 'transformers for text understanding'

1. [Score: 0.466] ██████████████████
   BERT: Pre-training of Deep Bidirectional Transformers

2. [Score: 0.316] ████████████
   Attention Mechanisms in Natural Language Processing

3. [Score: 0.146] █████
   Deep Learning for Image Classification Using Convolutional Neural Networks

🔍 Query: 'environmental impact of oceans'

1. [Score: 0.668] ██████████████████████████
   Climate Change Impact on Marine Ecosystems

2. [Score: 0.504] ████████████████████
   Ocean Acidification and Coral Reef Degradation

3. [Score: 0.240] █████████
   Cryptocurrency Mining and Environmental Sustainability

🔍 Query: 'medical AI applications'

1. [Score: 0.637] █████████████████████████
   AI-Based Disease Diagnosis Using Medical Imaging

2. [Score: 0.458] ██████████████████
   Machine Learning in Healthcare: Predictive Analytics

3. [Score: 0.235] █████████
   Deep Learning for Image Classification Using Convolutional Neural Networks
